# Excel / CSV → GeoJSON 変換
必須列は `name` / `latitude` / `longitude` の3つです。

## 1. ファイルをアップロード
Colab左側の **Files** からExcelまたはCSVをアップロードします。

In [ ]:
import pandas as pd
import json
from pathlib import Path

FILE_NAME = "sample.xlsx"  # ← 自分のファイル名に変更
SHEET_NAME = 0

## 2. データを読み込む

In [ ]:
path = Path(FILE_NAME)
df = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path, sheet_name=SHEET_NAME)
df.head()

## 3. 必須の列を確認する

In [ ]:
print(df.columns.tolist())
required = ["name", "latitude", "longitude"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"必要な列がありません: {missing}")
print("OK! 必要な列が揃っています。")

## 4. GeoJSONに変換する
緯度・経度は `geometry` に、それ以外の列はすべて `properties` に入れます。

In [ ]:
features = []

for _, row in df.iterrows():
    if pd.isna(row["latitude"]) or pd.isna(row["longitude"]):
        continue

    properties = {}
    for column in df.columns:
        if column not in ["latitude", "longitude"]:
            value = row[column]
            properties[column] = None if pd.isna(value) else value

    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["longitude"]), float(row["latitude"])]
        },
        "properties": properties
    })

geojson = {"type": "FeatureCollection", "features": features}
print(f"{len(features)} 件をGeoJSONに変換しました。")

## 5. 1件だけ中身を確認する

In [ ]:
print(json.dumps(geojson["features"][0], ensure_ascii=False, indent=2))

## 6. GeoJSONファイルを保存する

In [ ]:
OUTPUT_FILE = "mapdata.geojson"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2, allow_nan=False)
print(f"{OUTPUT_FILE} を作成しました！")

## 7. 地図で確認する
FoliumはLeafletを利用して、Colab上にインタラクティブな地図を表示できます。

In [ ]:
import folium

center_lat = df["latitude"].astype(float).mean()
center_lon = df["longitude"].astype(float).mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

folium.GeoJson(
    geojson,
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["名前:"])
).add_to(m)

m

## 完成！
次はデータや表示を変えて、自分だけのMAPにカスタマイズしてみましょう。